In [ ]:
# !pip install faiss-cpu -q
# !pip install -U torchao -q
import torch
import numpy as np
import random
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from sentence_transformers import SentenceTransformer
import faiss
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

In [ ]:
# =========================================================
# Cell 2: Fixed config (locked from validation runs)
# =========================================================
MODEL_NAME = "google/flan-t5-small"
DATASET_NAME = "databricks/databricks-dolly-15k"
SUBSET_SIZE = 1000
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128

PER_DEVICE_TRAIN_BS = 3
GRAD_ACCUM_STEPS = 2
PER_DEVICE_EVAL_BS = 2
MAX_STEPS = 300
LEARNING_RATE = 3e-3


# Set TOP_K to match whatever 3k/5k actually used, so 1k is now consistent with them.
TOP_K = 8
EMBED_INSTRUCTION_ONLY = False

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

RESULTS_LOG_PATH = "./group1_results.json"

In [ ]:
# =========================================================
# Cell 3: Load and format data (run once, reused across all 12 runs)
# =========================================================
print("Loading dataset...")
raw_dataset = load_dataset(DATASET_NAME, split="train")
raw_dataset = raw_dataset.shuffle(seed=42).select(range(SUBSET_SIZE))

def format_example(example):
    if example.get("context"):
        prompt = f"Instruction: {example['instruction']}\nContext: {example['context']}"
    else:
        prompt = f"Instruction: {example['instruction']}"
    return {"input_text": prompt, "target_text": example["response"]}

raw_dataset = raw_dataset.map(format_example)

# Fixed train/eval split, same for every run in this group (only strategy/seed varies)
split = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(example):
    model_inputs = tokenizer(
        example["input_text"], max_length=MAX_INPUT_LEN, truncation=True, padding="max_length",
    )
    labels = tokenizer(
        text_target=example["target_text"], max_length=MAX_TARGET_LEN, truncation=True, padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
eval_tokenized = eval_dataset.map(preprocess, remove_columns=eval_dataset.column_names)

Loading dataset...
Train size: 900, Eval size: 100


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# =========================================================
# Cell 4: Build semantic embeddings + FAISS index (run once)
# =========================================================
print("Building embeddings for semantic grouping...")
embedder = SentenceTransformer(EMBED_MODEL_NAME)

def get_embed_text(example):
    if EMBED_INSTRUCTION_ONLY:
        return example["instruction"]
    else:
        if example.get("context"):
            return f"{example['instruction']} {example['context']}"
        return example["instruction"]

embed_texts = [get_embed_text(ex) for ex in train_dataset]
embeddings = embedder.encode(embed_texts, show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])  # cosine similarity via inner product on normalized vectors
index.add(embeddings)
N = len(train_dataset)
print(f"FAISS index built: {N} vectors, dim={embeddings.shape[1]}")

Building embeddings for semantic grouping...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

FAISS index built: 900 vectors, dim=384


In [ ]:
# =========================================================
# Cell 5: Batch order construction functions
# =========================================================

def build_random_order(n_batches, seed):
    """Returns a flat list of indices, length n_batches * PER_DEVICE_TRAIN_BS,
    each batch of PER_DEVICE_TRAIN_BS drawn independently at random from the dataset."""
    rng = np.random.RandomState(seed)
    order = []
    for _ in range(n_batches):
        batch = rng.choice(N, size=PER_DEVICE_TRAIN_BS, replace=False)
        order.extend(batch.tolist())
    return order

def build_grouped_order(n_batches, seed):
    """Returns a flat list of indices where each consecutive PER_DEVICE_TRAIN_BS-sized
    chunk is a semantically similar group: an anchor + its (PER_DEVICE_TRAIN_BS - 1)
    nearest neighbors via FAISS, using TOP_K as the neighbor pool to sample from."""
    rng = np.random.RandomState(seed)
    order = []
    anchor_pool = list(range(N))
    rng.shuffle(anchor_pool)
    pool_idx = 0
    for _ in range(n_batches):
        if pool_idx >= len(anchor_pool):
            rng.shuffle(anchor_pool)
            pool_idx = 0
        anchor = anchor_pool[pool_idx]
        pool_idx += 1
        query_vec = embeddings[anchor:anchor+1]
        _, neighbor_ids = index.search(query_vec, TOP_K + 1)  # +1 because anchor itself is included
        neighbor_ids = [i for i in neighbor_ids[0] if i != anchor][:TOP_K]
        chosen = rng.choice(neighbor_ids, size=min(PER_DEVICE_TRAIN_BS - 1, len(neighbor_ids)), replace=False)
        batch = [anchor] + chosen.tolist()
        while len(batch) < PER_DEVICE_TRAIN_BS:
            batch.append(int(rng.choice(N)))
        order.extend(batch)
    return order

def build_curriculum_order(strategy, n_batches, seed):
    """grouped_to_random: first half of batches grouped, second half random.
    random_to_grouped: first half random, second half grouped."""
    half = n_batches // 2
    if strategy == "grouped_to_random":
        first = build_grouped_order(half, seed)
        second = build_random_order(n_batches - half, seed + 1000)
    elif strategy == "random_to_grouped":
        first = build_random_order(half, seed)
        second = build_grouped_order(n_batches - half, seed + 1000)
    else:
        raise ValueError(strategy)
    return first + second

In [ ]:
# =========================================================
# Cell 6: Custom sampler + Trainer subclass to enforce exact batch order
# =========================================================

class FixedOrderSampler(torch.utils.data.Sampler):
    def __init__(self, indices):
        self.indices = indices
    def __iter__(self):
        return iter(self.indices)
    def __len__(self):
        return len(self.indices)

class OrderedTrainer(Seq2SeqTrainer):
    def __init__(self, *args, fixed_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.fixed_order = fixed_order

    def get_train_dataloader(self):
        sampler = FixedOrderSampler(self.fixed_order)
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            drop_last=True,
        )

In [ ]:
# =========================================================
# Cell 7: Single-run function
# =========================================================

def run_single_experiment(strategy, seed):
    print(f"\n{'='*60}\nSTRATEGY={strategy}  SEED={seed}\n{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Fresh model load - required every run
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM, r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q", "v"],
    )
    model = get_peft_model(model, lora_config)

    # Number of physical batches needed to reach MAX_STEPS optimizer updates
    # (accounting for gradient accumulation)
    n_batches = MAX_STEPS * GRAD_ACCUM_STEPS

    if strategy == "random":
        order = build_random_order(n_batches, seed)
    elif strategy == "grouped":
        order = build_grouped_order(n_batches, seed)
    elif strategy == "grouped_to_random":
        order = build_curriculum_order("grouped_to_random", n_batches, seed)
    elif strategy == "random_to_grouped":
        order = build_curriculum_order("random_to_grouped", n_batches, seed)
    else:
        raise ValueError(strategy)

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./output_{strategy}_{seed}",
        per_device_train_batch_size=PER_DEVICE_TRAIN_BS,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BS,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=50,
        eval_strategy="no",       # evaluate manually at the end to save time across 12 runs
        save_strategy="no",
        seed=seed,
        report_to="none",
        predict_with_generate=True,
        fp16=False,
    )

    trainer = OrderedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=eval_tokenized,
        data_collator=data_collator,
        processing_class=tokenizer,
        fixed_order=order,
    )

    trainer.train()
    eval_results = trainer.evaluate()
    eval_loss = eval_results.get("eval_loss")
    print(f"RESULT  strategy={strategy}  seed={seed}  eval_loss={eval_loss}")

    # --- Generation + ROUGE/BERTScore ---
    # trainer.evaluate() only returns loss; generation metrics require actually
    # generating text and comparing it to the reference responses.
    print("Generating outputs for ROUGE/BERTScore...")
    model.eval()
    predictions = []
    references = [eval_dataset[i]["target_text"] for i in range(len(eval_dataset))]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    GEN_BATCH_SIZE = 64  # batched generation - much faster than one-at-a-time

    with torch.no_grad():
        for start in range(0, len(eval_dataset), GEN_BATCH_SIZE):
            end = min(start + GEN_BATCH_SIZE, len(eval_dataset))
            batch_input_ids = torch.tensor(
                [eval_tokenized[i]["input_ids"] for i in range(start, end)]
            ).to(device)
            batch_attention_mask = torch.tensor(
                [eval_tokenized[i]["attention_mask"] for i in range(start, end)]
            ).to(device)
            generated = model.generate(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                max_new_tokens=MAX_TARGET_LEN,
            )
            batch_preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
            predictions.extend(batch_preds)

    # ROUGE-1/2/L
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores["rouge1"].fmeasure)
        rouge2_scores.append(scores["rouge2"].fmeasure)
        rougeL_scores.append(scores["rougeL"].fmeasure)

    rouge1 = sum(rouge1_scores) / len(rouge1_scores)
    rouge2 = sum(rouge2_scores) / len(rouge2_scores)
    rougeL = sum(rougeL_scores) / len(rougeL_scores)


    print(f"RESULT  strategy={strategy}  seed={seed}  "
      f"ROUGE-1={rouge1:.4f}  ROUGE-2={rouge2:.4f}  ROUGE-L={rougeL:.4f}")

    metrics = {
        "eval_loss": eval_loss,
        "rouge1": rouge1,
        "rouge2": rouge2,
        "rougeL": rougeL,
    }

    # free memory before next run
    del model, trainer
    torch.cuda.empty_cache()

    return metrics

In [ ]:
# !pip install -U bert_score -q

In [ ]:
# !pip install transformers -U -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 34.2 MB/s eta 0:00:00


In [ ]:
# =========================================================
# Cell 8: Run all 12 experiments (Group 1)
# =========================================================

STRATEGIES = ["random", "grouped", "grouped_to_random", "random_to_grouped"]
SEEDS = [13, 21, 42]

results = []

for strategy in STRATEGIES:
    for seed in SEEDS:
        metrics = run_single_experiment(strategy, seed)
        results.append({"strategy": strategy, "seed": seed, **metrics})
        # save incrementally in case of disconnect
        with open(RESULTS_LOG_PATH, "w") as f:
            json.dump(results, f, indent=2)

print("\n\nALL GROUP 1 RUNS COMPLETE")
for r in results:
    print(r)


STRATEGY=random  SEED=13


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Step,Training Loss
50,19.136350
100,7.333892
150,6.377709
200,5.881171
250,5.486197
300,5.479628


Training Loss,Validation Loss,Step
5.479628,2.491051,300


...

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Step,Training Loss
50,17.468318
100,7.332560
150,6.461315
200,5.913830
250,5.702894
300,5.594609


Training Loss,Validation Loss,Step
5.594609,2.515094,300


RESULT  strategy=random_to_grouped  seed=42  eval_loss=2.5150938034057617
Generating outputs for ROUGE/BERTScore...
RESULT  strategy=random_to_grouped  seed=42  ROUGE-1=0.2616  ROUGE-2=0.1172  ROUGE-L=0.2326


ALL GROUP 1 RUNS COMPLETE
{'strategy': 'random', 'seed': 13, 'eval_loss': 2.491051435470581, 'rouge1': 0.24048353377902376, 'rouge2': 0.1078602899360381, 'rougeL': 0.20399759759557395}
{'strategy': 'random', 'seed': 21, 'eval_loss': 2.5587451457977295, 'rouge1': 0.2520428851924599, 'rouge2': 0.1121252088983317, 'rougeL': 0.2144895737675266}
{'strategy': 'random', 'seed': 42, 'eval_loss': 2.5223658084869385, 'rouge1': 0.22462975528939058, 'rouge2': 0.09399575102098227, 'rougeL': 0.19814213989509138}
{'strategy': 'grouped', 'seed': 13, 'eval_loss': 2.4721570014953613, 'rouge1': 0.24146035624254175, 'rouge2': 0.10506485546767702, 'rougeL': 0.20610632610352916}
{'strategy': 'grouped', 'seed': 21, 'eval_loss': 2.532271385192871, 'rouge1': 0.2664770335092741, 'rouge2': 0.12107115259712